# STAT 8017B Project 4 — Unsupervised Learning & Text Analytics
## Financial Analysis Chatbot (Group 4.1)

This notebook covers:

1. **Descriptive Statistics & Correlation** — fund data
2. **PCA** — dimensionality reduction on fund features
3. **K-Means Clustering** — cluster funds by performance profile
4. **Association Rules** (Apriori) — mine patterns in complaint attributes
5. **LSA** (Latent Semantic Analysis) — extract latent topics from financial text

**Course methods**: Descriptive stats, Correlation (Ch.2), PCA (Ch.5), K-Means (Tutorial 5), Association Rules (Tutorial 6), LSA/TruncatedSVD (Ch.2)

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import TfidfVectorizer

from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

warnings.filterwarnings('ignore')

PROCESSED_DIR = os.path.join('..', 'data', 'processed')
MODEL_DIR = os.path.join('..', 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

print('Setup complete.')

---
## 1. Descriptive Statistics & Correlation Analysis

Detailed statistics on ETF and Mutual Fund data: mean, median, std, distributions.
Pearson correlation between expense ratio and returns.

In [ ]:
etf = pd.read_csv(os.path.join(PROCESSED_DIR, 'etf_clean.csv'))
mf = pd.read_csv(os.path.join(PROCESSED_DIR, 'mutualfund_clean.csv'))

etf['source'] = 'ETF'
mf['source'] = 'MutualFund'
common_cols = list(set(etf.columns) & set(mf.columns))
funds = pd.concat([etf[common_cols], mf[common_cols]], ignore_index=True)

numeric_cols = funds.select_dtypes(include=[np.number]).columns.tolist()
print(f'Combined funds: {len(funds):,} rows')
print(f'\nDescriptive Statistics:')
funds[numeric_cols].describe().round(4)

In [ ]:
return_cols = [c for c in numeric_cols if 'return' in c.lower()]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, col in enumerate(return_cols[:6]):
    ax = axes[i // 3, i % 3]
    funds[col].dropna().hist(bins=50, ax=ax, edgecolor='black', alpha=0.7)
    ax.set_title(col.replace('fund_', '').replace('_', ' ').title())
    ax.axvline(funds[col].median(), color='red', linestyle='--', label=f'Median: {funds[col].median():.2f}')
    ax.legend(fontsize=8)
plt.suptitle('Fund Return Distributions', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix heatmap
corr_cols = [c for c in numeric_cols if any(k in c for k in ['return', 'expense', 'net_assets', 'yield', 'sharpe', 'beta', 'stdev'])]
corr_matrix = funds[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
short_labels = [c.replace('fund_', '').replace('annual_report_net_', '') for c in corr_cols]
ax.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(short_labels, fontsize=8)

for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}', ha='center', va='center', fontsize=6)

plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('Pearson Correlation Matrix — Fund Features')
plt.tight_layout()
plt.show()

In [ ]:
# Expense ratio vs 1-year return scatter with regression line
exp_col = 'fund_annual_report_net_expense_ratio'
ret_col = 'fund_return_1year'

if exp_col in funds.columns and ret_col in funds.columns:
    scatter_df = funds[[exp_col, ret_col]].dropna()
    r_val = scatter_df[exp_col].corr(scatter_df[ret_col])

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(scatter_df[exp_col], scatter_df[ret_col], alpha=0.2, s=5)
    z = np.polyfit(scatter_df[exp_col], scatter_df[ret_col], 1)
    p = np.poly1d(z)
    x_line = np.linspace(scatter_df[exp_col].min(), scatter_df[exp_col].max(), 100)
    ax.plot(x_line, p(x_line), 'r-', linewidth=2, label=f'r = {r_val:.3f}')
    ax.set_xlabel('Expense Ratio')
    ax.set_ylabel('1-Year Return')
    ax.set_title('Expense Ratio vs 1-Year Return')
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f'Pearson correlation (expense ratio vs 1yr return): r = {r_val:.4f}')

---
## 2. PCA — Dimensionality Reduction on Fund Features

Reduce the fund feature space and visualize explained variance.

In [ ]:
pca_cols = [c for c in numeric_cols if c in funds.columns]
funds_pca_df = funds[pca_cols].dropna()
print(f'PCA input: {funds_pca_df.shape[0]:,} rows, {funds_pca_df.shape[1]} features')

scaler_pca = StandardScaler()
X_pca_scaled = scaler_pca.fit_transform(funds_pca_df)

pca_full = PCA(random_state=42)
pca_full.fit(X_pca_scaled)

# Explained variance plot
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, len(pca_full.explained_variance_ratio_) + 1),
            pca_full.explained_variance_ratio_, alpha=0.7)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained')
axes[0].set_title('Individual Explained Variance')

axes[1].plot(range(1, len(cumvar) + 1), cumvar, 'bo-')
axes[1].axhline(0.90, color='red', linestyle='--', label='90% threshold')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Variance Explained')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend()

plt.tight_layout()
plt.show()

n_90 = np.argmax(cumvar >= 0.90) + 1
print(f'Components needed for 90% variance: {n_90}')

In [ ]:
# 2D PCA projection
pca_2d = PCA(n_components=2, random_state=42)
X_2d = pca_2d.fit_transform(X_pca_scaled)

fig, ax = plt.subplots(figsize=(10, 7))
# Color by source if available
source_col = funds.loc[funds_pca_df.index, 'source'] if 'source' in funds.columns else None
if source_col is not None:
    for src, color in [('ETF', '#3498db'), ('MutualFund', '#e74c3c')]:
        mask = source_col == src
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1], alpha=0.3, s=5, c=color, label=src)
    ax.legend()
else:
    ax.scatter(X_2d[:, 0], X_2d[:, 1], alpha=0.3, s=5)

ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%})')
ax.set_title('PCA 2D Projection — ETFs vs Mutual Funds')
plt.tight_layout()
plt.show()

---
## 3. K-Means Clustering — Fund Segmentation

Cluster funds by performance profile. Use elbow method + silhouette score to pick K.

In [ ]:
K_range = range(2, 11)
inertias = []
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    labels = km.fit_predict(X_pca_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_pca_scaled, labels, sample_size=min(5000, len(X_pca_scaled)))
    silhouettes.append(sil)
    print(f'K={k:2d}  Inertia={km.inertia_:12.0f}  Silhouette={sil:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(K_range), inertias, 'bo-')
axes[0].set_xlabel('K')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')

axes[1].plot(list(K_range), silhouettes, 'ro-')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score vs K')

plt.tight_layout()
plt.show()

best_k = list(K_range)[np.argmax(silhouettes)]
print(f'\nBest K by silhouette: {best_k}')

In [ ]:
# Final clustering with best K
kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = kmeans_final.fit_predict(X_pca_scaled)

funds_clustered = funds.loc[funds_pca_df.index].copy()
funds_clustered['cluster'] = cluster_labels

# Visualize clusters on PCA 2D
fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=cluster_labels, cmap='tab10', alpha=0.4, s=5)

centers_2d = pca_2d.transform(scaler_pca.transform(
    pd.DataFrame(kmeans_final.cluster_centers_, columns=pca_cols)
))
ax.scatter(centers_2d[:, 0], centers_2d[:, 1], c='black', marker='X', s=200, edgecolors='white', linewidth=2)

ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%})')
ax.set_title(f'K-Means Clustering (K={best_k}) on PCA Projection')
plt.colorbar(scatter, ax=ax, label='Cluster')
plt.tight_layout()
plt.show()

# Cluster profiles
profile_cols = [c for c in ['fund_return_1year', 'fund_annual_report_net_expense_ratio',
                             'total_net_assets', 'fund_sharpe_ratio_3years',
                             'fund_stdev_3years'] if c in funds_clustered.columns]
print('\nCluster Profiles (mean values):')
print(funds_clustered.groupby('cluster')[profile_cols].mean().round(4).to_string())

joblib.dump(kmeans_final, os.path.join(MODEL_DIR, 'fund_kmeans.joblib'))
joblib.dump(scaler_pca, os.path.join(MODEL_DIR, 'fund_pca_scaler.joblib'))

---
## 4. Association Rules — Complaint Attribute Patterns

Use Apriori algorithm to find frequent co-occurrences among complaint attributes
(Product, Issue, State).

In [ ]:
comp = pd.read_csv(os.path.join(PROCESSED_DIR, 'complaints_clean.csv'))
print(f'Complaints: {len(comp):,} rows')

# Build transactions from categorical attributes
transactions = []
for _, row in comp.iterrows():
    items = []
    if pd.notna(row.get('Product')):
        items.append(f"Product={row['Product']}")
    if pd.notna(row.get('Issue')):
        items.append(f"Issue={row['Issue']}")
    if pd.notna(row.get('State')):
        items.append(f"State={row['State']}")
    if items:
        transactions.append(items)

print(f'Transactions built: {len(transactions):,}')
print(f'Example: {transactions[0]}')

In [ ]:
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_array, columns=te.columns_)
print(f'One-hot encoded: {df_encoded.shape}')

frequent_items = apriori(df_encoded, min_support=0.02, use_colnames=True)
print(f'Frequent itemsets found: {len(frequent_items)}')
frequent_items.sort_values('support', ascending=False).head(20)

In [ ]:
if len(frequent_items) > 0:
    rules = association_rules(frequent_items, metric='lift', min_threshold=1.2)
    rules = rules.sort_values('lift', ascending=False)
    print(f'Association rules found: {len(rules)}')
    print('\nTop 20 rules by lift:')
    display_cols = ['antecedents', 'consequents', 'support', 'confidence', 'lift']
    print(rules[display_cols].head(20).to_string(index=False))

    # Scatter: support vs confidence, colored by lift
    fig, ax = plt.subplots(figsize=(10, 6))
    sc = ax.scatter(rules['support'], rules['confidence'],
                    c=rules['lift'], cmap='YlOrRd', alpha=0.6, s=30)
    plt.colorbar(sc, ax=ax, label='Lift')
    ax.set_xlabel('Support')
    ax.set_ylabel('Confidence')
    ax.set_title('Association Rules — Support vs Confidence (color = Lift)')
    plt.tight_layout()
    plt.show()
else:
    print('No frequent itemsets found. Try lowering min_support.')

---
## 5. LSA (Latent Semantic Analysis) — Topic Extraction

Apply TruncatedSVD to TF-IDF matrix to discover latent topics in financial news.

In [ ]:
finsen = pd.read_csv(os.path.join(PROCESSED_DIR, 'finsen_clean.csv'))
print(f'FinSen: {len(finsen):,} rows')

tfidf_lsa = TfidfVectorizer(max_features=5000)
X_lsa = tfidf_lsa.fit_transform(finsen['cleaned_content'])
print(f'TF-IDF matrix: {X_lsa.shape}')

feature_names = tfidf_lsa.get_feature_names_out()

In [ ]:
N_TOPICS = 10

lsa = TruncatedSVD(n_components=N_TOPICS, random_state=42)
X_lsa_topics = lsa.fit_transform(X_lsa)

print(f'Explained variance ratio: {lsa.explained_variance_ratio_.sum():.2%}\n')

for i, component in enumerate(lsa.components_):
    top_indices = component.argsort()[-10:][::-1]
    top_words = [feature_names[j] for j in top_indices]
    print(f'Topic {i+1}: {" | ".join(top_words)}')

In [ ]:
# Visualize documents in 2D LSA space, colored by category
le_cat = joblib.load(os.path.join(PROCESSED_DIR, 'finsen_label_encoder.joblib'))

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(X_lsa_topics[:, 0], X_lsa_topics[:, 1],
                     c=finsen['category_label'], cmap='tab20', alpha=0.4, s=5)
ax.set_xlabel('LSA Component 1')
ax.set_ylabel('LSA Component 2')
ax.set_title('LSA 2D Projection of Financial News — Colored by Category')
plt.colorbar(scatter, ax=ax, label='Category')
plt.tight_layout()
plt.show()

In [ ]:
# Also apply LSA to complaints text
X_comp_tfidf = joblib.load(os.path.join(PROCESSED_DIR, 'complaints_tfidf_matrix.joblib'))
tfidf_comp = joblib.load(os.path.join(PROCESSED_DIR, 'complaints_tfidf.joblib'))

lsa_comp = TruncatedSVD(n_components=10, random_state=42)
X_comp_lsa = lsa_comp.fit_transform(X_comp_tfidf)

comp_features = tfidf_comp.get_feature_names_out()
print('LSA Topics from Complaint Text:\n')
for i, component in enumerate(lsa_comp.components_):
    top_indices = component.argsort()[-10:][::-1]
    top_words = [comp_features[j] for j in top_indices]
    print(f'Topic {i+1}: {" | ".join(top_words)}')

print(f'\nExplained variance: {lsa_comp.explained_variance_ratio_.sum():.2%}')

---
## Summary

Unsupervised learning and text analytics complete:
- Descriptive stats and correlation for fund data
- PCA reveals key variance dimensions
- K-Means segments funds into distinct clusters
- Association rules reveal product-issue-state patterns in complaints
- LSA extracts meaningful latent topics from financial text